# Multi-Agent Workflow for Biomedical Literature Review

## 1. Setup & Data Access

Install dependencies, configure S3 client and E-utilities, search PMC for articles (3 test queries), fetch XMLs from S3, parse XML → structured JSON, and run quality checks.

### Install Dependencies

Run once to install all required packages.

In [79]:
%pip install boto3 requests lxml chromadb sentence-transformers transformers torch pandas jupyter

Defaulting to user installation because normal site-packages is not writeable
Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip is available: 25.3 -> 26.0.1
[notice] To update, run: C:\Users\User\AppData\Local\Microsoft\WindowsApps\PythonSoftwareFoundation.Python.3.12_qbz5n2kfra8p0\python.exe -m pip install --upgrade pip


### Imports

In [80]:
import requests
import time
import json
import boto3
from botocore import UNSIGNED
from botocore.config import Config
from lxml import etree

### Step 1 — Article Discovery via PubMed E-utilities

Use NCBI's E-utilities API to search PubMed Central by topic and retrieve PMC IDs.

In [81]:
def search_pmc(query: str, max_results: int = 30) -> list[str]:
    """Search PMC and return list of PMC IDs."""
    url = "https://eutils.ncbi.nlm.nih.gov/entrez/eutils/esearch.fcgi"
    params = {
        "db": "pmc",
        "term": query,
        "retmax": max_results,
        "retmode": "json"
    }
    response = requests.get(url, params=params)
    response.raise_for_status()
    data = response.json()
    pmc_ids = data["esearchresult"]["idlist"]
    return [f"PMC{pid}" for pid in pmc_ids]

In [82]:
# Test queries from the plan
test_queries = [
    "mRNA vaccine adverse events pediatric",
    "Transformer-based models for protein folding",
    "Clinical trial outcomes for monoclonal antibodies in oncology"
]

# Search for all 3 queries, collect PMC IDs, deduplicate
all_pmc_ids = set()
for query in test_queries:
    print(f"Searching: {query}")
    pmc_ids = search_pmc(query, max_results=30) # take top 30 results for each query for now
    all_pmc_ids.update(pmc_ids) 
    print(f"  Found {len(pmc_ids)} IDs")
    time.sleep(0.5)  # Rate limiting: 3 req/sec without API key

all_pmc_ids = list(all_pmc_ids)
print(f"\nTotal unique PMC IDs: {len(all_pmc_ids)}")

Searching: mRNA vaccine adverse events pediatric
  Found 30 IDs
Searching: Transformer-based models for protein folding
  Found 30 IDs
Searching: Clinical trial outcomes for monoclonal antibodies in oncology
  Found 30 IDs

Total unique PMC IDs: 90


In [83]:
# test the first query (in slightly nicer form) to ensure we get results
import requests
url = "https://eutils.ncbi.nlm.nih.gov/entrez/eutils/esearch.fcgi"
params = {
    "db": "pmc",
    "term": "mRNA vaccine adverse events pediatric",
    "retmax": 30,
    "retmode": "json"
}
r = requests.get(url, params=params)
print(r.json()["esearchresult"]["count"])

10686


### Step 2 — Fetch XML Files from S3

Download full-text XML for each PMC ID from the public PubMed Central Open Access bucket.

In [84]:
# S3 client for public bucket (no AWS credentials needed for now)
s3 = boto3.client(
    "s3",
    region_name="us-east-1",
    config=Config(signature_version=UNSIGNED), # without credentials
)


# fetch function for an article xml from S3
def fetch_xml_from_s3(pmc_id: str) -> str | None:
    """Download XML for a given PMC ID from S3, trying both old and new paths."""
    # New structure (for newer articles): PMC12345678.1/PMC12345678.1.xml
    new_key = f"{pmc_id}.1/{pmc_id}.1.xml"
    # Old structure (available until Aug 2026): oa_comm/xml/all/PMC12345678.xml
    old_key = f"oa_comm/xml/all/{pmc_id}.xml"
    
    for key in [new_key, old_key]:
        try:
            obj = s3.get_object(Bucket="pmc-oa-opendata", Key=key)
            return obj["Body"].read().decode("utf-8")
        except Exception:
            continue
    
    print(f"WARNING: {pmc_id} not found in oa_comm. Skipping.")
    return None


In [85]:
# Fetch XML for all PMC IDs
raw_xmls = []
for i, pmc_id in enumerate(all_pmc_ids):
    xml = fetch_xml_from_s3(pmc_id)
    if xml is not None:
        raw_xmls.append(xml)
    if (i + 1) % 10 == 0:
        print(f"Fetched {i + 1}/{len(all_pmc_ids)}...")

print(f"\nSuccessfully fetched {len(raw_xmls)} XML files")

Fetched 10/90...
Fetched 20/90...
Fetched 30/90...
Fetched 40/90...
Fetched 50/90...
Fetched 60/90...
Fetched 70/90...
Fetched 80/90...
Fetched 90/90...

Successfully fetched 82 XML files


### Step 3 — Parse XML to Structured JSON

Parse each JATS XML file and extract structured fields.

In [86]:
def parse_article_xml(xml_string: str) -> dict | None:
    """Parse JATS XML and return structured article dict."""
    try:
        root = etree.fromstring(xml_string.encode("utf-8"))
    except etree.XMLSyntaxError:
        return None

    def get_text(element):
        """Recursively get all text from an element and its children."""
        return "".join(element.itertext()).strip() if element is not None else ""

    front = root.find(".//front")
    article_meta = front.find(".//article-meta") if front is not None else None

    if article_meta is None:
        return None

    # PMC ID
    pmc_id = ""
    for aid in article_meta.findall(".//article-id"):
        if aid.get("pub-id-type") == "pmc":
            pmc_id = aid.text or ""
            break

    # PMID
    pmid = ""
    for aid in article_meta.findall(".//article-id"):
        if aid.get("pub-id-type") == "pmid":
            pmid = aid.text or ""
            break

    # Title
    title_el = article_meta.find(".//article-title")
    title = get_text(title_el)

    # Abstract
    abstract_el = article_meta.find(".//abstract")
    abstract = get_text(abstract_el) if abstract_el is not None else ""

    # Authors
    authors = []
    for contrib in article_meta.findall(".//contrib[@contrib-type='author']"):
        surname = contrib.findtext(".//surname", default="")
        given = contrib.findtext(".//given-names", default="")
        if surname:
            authors.append(f"{given} {surname}".strip())

    # Journal
    journal = ""
    journal_el = front.find(".//journal-title") if front is not None else None
    if journal_el is not None:
        journal = get_text(journal_el)

    # Publication date
    pub_date = ""
    pub_date_el = article_meta.find(".//pub-date")
    if pub_date_el is not None:
        year = pub_date_el.findtext("year", default="")
        month = pub_date_el.findtext("month", default="")
        day = pub_date_el.findtext("day", default="")
        pub_date = f"{year}-{month.zfill(2)}-{day.zfill(2)}" if year else ""

    # Keywords
    keywords = []
    for kwd in article_meta.findall(".//kwd"):
        kw_text = get_text(kwd)
        if kw_text:
            keywords.append(kw_text)

    # Body text (for future RAG use)
    body_el = root.find(".//body")
    body_text = get_text(body_el) if body_el is not None else ""

    # Normalize PMC ID format (ensure PMC prefix)
    if pmc_id and not pmc_id.upper().startswith("PMC"):
        pmc_id = f"PMC{pmc_id}"

    return {
        "pmc_id": pmc_id,
        "pmid": pmid,
        "title": title,
        "abstract": abstract,
        "authors": authors,
        "journal": journal,
        "pub_date": pub_date,
        "keywords": keywords,
        "body_text": body_text[:5000]
    }

In [87]:
# Parse all XMLs and filter out articles without abstracts
articles = [parse_article_xml(xml) for xml in raw_xmls if xml is not None]
articles = [a for a in articles if a is not None and a["abstract"]] # keep only those with abstracts

# Save to JSON
with open("articles.json", "w") as f:
    json.dump(articles, f, indent=2)

print(f"Saved {len(articles)} articles to articles.json")

Saved 82 articles to articles.json


### Quality Check

Print stats: total articles, with abstracts, with keywords.

In [96]:
total = len(articles)
with_abstracts = sum(1 for a in articles if a.get("abstract"))
with_keywords = sum(1 for a in articles if a.get("keywords"))

print("=== Quality Check ===")
print(f"Total articles:        {total}")
print(f"With abstracts:       {with_abstracts}")
print(f"With keywords:        {with_keywords}")
print(f"Without abstracts:    {total - with_abstracts} (excluded from index)")

=== Quality Check ===
Total articles:        82
With abstracts:       82
With keywords:        65
Without abstracts:    0 (excluded from index)


## 2. Retriever Agent

Load articles from JSON, build ChromaDB index with all-MiniLM-L6-v2 embeddings, and define the RetrieverAgent for semantic search.

### Step 4 — Build Embeddings & ChromaDB Index

Load parsed articles, generate embeddings with sentence-transformers (all-MiniLM-L6-v2), and store in ChromaDB with cosine similarity.

In [ ]:
import chromadb
from chromadb.utils import embedding_functions

# Load articles (from JSON if not already in memory from previous data acquisition step)
try:
    articles
except NameError:
    with open("articles.json", "r") as f:
        articles = json.load(f)

# Sentence-transformers embedding function
ef = embedding_functions.SentenceTransformerEmbeddingFunction(
    model_name="sentence-transformers/all-MiniLM-L6-v2"
)

# In-memory client (sufficient for <100 articles)
client = chromadb.Client()

# use get_or_create_collection to avoid errors if we rerun this cell multiple times during development
collection = client.get_or_create_collection(
    name="pmc_articles",
    embedding_function=ef,
    metadata={"hnsw:space": "cosine"} # use cosine similarity search
)


# Prepare data for ChromaDB
documents = []
metadatas = []
ids = []

for article in articles:
    if not article.get("abstract"):
        continue
    
    # Use pmc_id, fall back to pmid, then to index
    article_id = article["pmc_id"] or f"PMID{article['pmid']}" or f"article_{len(ids)}"
    
    doc_text = f"{article['title']}. {article['abstract']}"
    documents.append(doc_text)
    metadatas.append({
        "pmc_id": article["pmc_id"],
        "pmid": article.get("pmid", ""),
        "title": article["title"],
        "journal": article.get("journal", ""),
        "pub_date": article.get("pub_date", ""),
        "keywords": ", ".join(article.get("keywords", [])),
    })
    ids.append(article_id)
    
collection.add(documents=documents, metadatas=metadatas, ids=ids)
print(f"Indexed {len(documents)} articles in ChromaDB.")

Indexed 82 articles in ChromaDB.


### Step 5 — Retriever Agent

Query ChromaDB and return top-K articles with full data. ChromaDB returns distance (lower = more similar); we convert to similarity as `1 - distance`.

In [101]:
class RetrieverAgent:
    """Retrieves top-K relevant articles from ChromaDB given a query."""

    def __init__(self, collection, articles_data: list, k: int = 5): # find top 5 here for testing
        self.collection = collection
        self.k = k
        self.articles_lookup = {}
        for a in articles_data:
            key = a["pmc_id"] or f"PMID{a['pmid']}" or f"article_unknown"
            self.articles_lookup[key] = a

    def retrieve(self, query: str, k: int = None) -> list[dict]:
        """Query ChromaDB and return top-K articles with full data."""
        k = k or self.k
        results = self.collection.query(query_texts=[query], n_results=k)

        retrieved = []
        ids_list = results["ids"][0] or []
        distances_list = results.get("distances", [[]])[0] if results.get("distances") else []

        for i in range(len(ids_list)):
            pmc_id = ids_list[i]
            distance = distances_list[i] if i < len(distances_list) else None
            article_data = self.articles_lookup.get(pmc_id, {})

            retrieved.append({
                "pmc_id": pmc_id,
                "pmid": article_data.get("pmid", ""),
                "title": article_data.get("title", ""),
                "abstract": article_data.get("abstract", ""),
                "authors": article_data.get("authors", []),
                "journal": article_data.get("journal", ""),
                "pub_date": article_data.get("pub_date", ""),
                "keywords": article_data.get("keywords", []),
                "relevance_score": round(1 - distance, 4) if distance is not None else None,
            })

        return retrieved

In [102]:
# Test Retriever Agent with all 3 queries
retriever = RetrieverAgent(collection, articles, k=5)

for query in test_queries:
    print(f"\n{'='*80}")
    print(f"Query: {query}")
    print(f"{'='*80}")
    results = retriever.retrieve(query)
    for i, r in enumerate(results):
        print(f"\n  [{i+1}] {r['title']}")
        print(f"      PMC: {r['pmc_id']} | Score: {r['relevance_score']}")
        print(f"      Journal: {r['journal']} | Date: {r['pub_date']}")


Query: mRNA vaccine adverse events pediatric

  [1] Association between nucleic acid COVID-19 vaccines and acute myocardial infarction in adults: a systematic review
      PMC: PMID41768578 | Score: 0.5409
      Journal: Frontiers in Cardiovascular Medicine | Date: 2026-02-12

  [2] Background incidence rates and observed-to-expected ratios of adverse events of special interest after covid-19 mRNA vaccination during pregnancy in France: a nationwide population-based study
      PMC: PMID41767889 | Score: 0.5375
      Journal: The Lancet Regional Health - Europe | Date: 2026-05-00

  [3] Avian Influenza H5N1 Infection During Pregnancy: Preparing for the Next Flu Pandemic and Improving Perinatal Outcomes
      PMC: PMID41754555 | Score: 0.4178
      Journal: Viruses | Date: 2026-02-06

  [4] Safety and health-related quality of life following maternal respiratory syncytial virus (RSV) and diphtheria–tetanus–acellular pertussis (DTaP) vaccinations
      PMC: PMID41744410 | Score: 0.4018


In [ ]:
# print(articles[0]["pmc_id"])
# print(articles[0]["title"])
# print(articles[0]["journal"])

# print(f"pmc_id: '{articles[0]['pmc_id']}'")
# print(f"pmid: '{articles[0]['pmid']}'")

# # Test 1: What IDs are actually in ChromaDB?
# print("First 5 ChromaDB IDs:", collection.peek(5)["ids"])

# # Test 2: What keys are in the lookup?
# print("First 5 lookup keys:", list(retriever.articles_lookup.keys())[:5])

# # Test 3: Do they match?
# chroma_ids = collection.peek(5)["ids"]
# for cid in chroma_ids:
#     print(f"ID '{cid}' in lookup: {cid in retriever.articles_lookup}")

# # Test 4: What does a raw query return from ChromaDB?
# raw = collection.query(query_texts=["mRNA vaccine"], n_results=2)
# print("Raw IDs:", raw["ids"])
# print("Raw metadatas:", raw["metadatas"])




Comparative efficacy and safety of S-1 plus oxaliplatin with sintilimab vs. paclitaxel–S-1–oxaliplatin and docetaxel–oxaliplatin–5-fluorouracil as first-line therapy for advanced gastric cancer
Anti-Cancer Drugs
pmc_id: ''
pmid: '41191583'
